# Νευρωνικά Δίκτυα Με Convolutional Επίπεδα

Έχουμε δεί την βασική λειτουργία των νευρωνικών σε [αυτό το Notebook](https://colab.research.google.com/drive/1EVNEGakrtSQTHcUdO_lYJPSIPMHEjJZb?usp=sharing), και πως τα εκπαιδεύουμε σε [αυτό το Notebook](https://colab.research.google.com/drive/1hCXEaMOxXnxWPoPn9YYufDF2huHUgHAS?usp=sharing). Τώρα περνάμε σε νευρωνικά δίκτυα με convolutional επίπεδα.

Πρίν περάσουμε στο CIFAR-10 και πιο δύσκολα προβλήματα, κοιτάμε ένα από τα "εύκολα" προβλήματα της Μηχανικής Όρασης: Το λεγόμενο MNIST -- αυτόματη αναγνώριση χειρόγραφων ψηφίων.


Ο σκοπός αυτού του Notebook είναι:
1.   Να συνεχίσουμε να μαθαίνουμε πως να ορίζουμε επίπεδα για να χτίσουμε βαθιά νευρωνικά δίκτυα.
2. Να μάθουμε πως να χρησιμοποιούμε **Convolutional**, **Maxpool** και **AveragePool** επίπεδα, και τι είναι.
  * Convolutional layers: `torch.nn.Conv2d`
  * Max Pooling Layers:`torch.nn.MaxPool2d`
  * Average Pooling Layers: `torch.nn.AvgPool2d`
3.   Να μάθουμε τις βασικές ιδέες της εκπαίδευσης για νευρωνικά δίκτυα (συνεχίζοντας αυτά που μάθαμε στο προηγούμενο Notebook).

Και θυμόμαστε τις βασικές ιδέες από το [προηγούμενο Notebook](https://colab.research.google.com/drive/1hCXEaMOxXnxWPoPn9YYufDF2huHUgHAS?usp=sharing): πως κάνουμε το "model = " και το "model.fit(X,y)" σε νευρωνικά δίκτυα. Είναι μεγαλύτερα τα δίκτυα που θα δούμε εδώ, όπως επίσης και τα δεδομένα.


Επίσης μπορείτε να δείτε:

https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html



```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np


In [2]:
# Set the random seed
import torch
torch.manual_seed(42)

# Κατεβάζουμε τα δεδομένα

In [3]:
# Downloading training and test datasets
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)



## Κοιτάμε τα Δεδομένα!

Πόσα έχουμε; Τι μορφή έχουν;

In [4]:
# Πρώτα να δούμε πόσα έχουμε
print(len(train_dataset))
print(len(test_dataset))

60000
10000


### Διαλέγουμε ένα

Η εντολή
```
image, label = train_dataset[130]
```
μας δίνει το "Χ" (image) και το "y" (label).

In [5]:
image, label = train_dataset[130]  # Select an image, you can change this index


### Τι περιέχει το "image"

Στην μορφή που τα κατεβάσαμε τα δεδομένα, τα χαρακτηριστικά εισόδου (features) είναι 28 x 28 = 784 αριθμοί από το 0 μέχρι το 1.

In [ ]:
print(image)
print(label)

**Εάν πολαπλασιάσουμε με 255** βλέπουμε πως τα features είναι αριθμοί από το 0 μέχρι το 255. Αυτοί αντιστοιχούν στην ένταση του μαυρόασπρου πίξελ.

Αυτό μπορούμε να το δούμε χρησιμοποιόντας το παρακάτω κώδικα.

In [ ]:

# Convert the tensor image to a numpy array and scale it from 0-1 to 0-255
image_np = (image.numpy().squeeze() * 255).astype(int)

# Plot the image with pixel intensities
plt.figure(figsize=(8, 8))
for i in range(image_np.shape[0]):
    for j in range(image_np.shape[1]):
        plt.text(j, i, str(image_np[i, j]), va='center', ha='center', fontsize=8, color="black")

plt.imshow(image_np, cmap="gray", interpolation="none")
plt.axis("off")
plt.show()

## Transforms

Όταν μεταφορτώνουμε οποιαδήποτε δεδομένα, πρέπει να έχουμε υπόψιν ποιούς αλγορίθμους σκοπεύουμε να χρησιμοποιήσουμε. Τα νευρωνικά δίκτυα δεν είναι γραμμικά, διότι όπως έχουμε δεί, περιέχουν μη-γραμμικές συναρτίσεις όπως το ReLU. Όμως, περιέχουν γραμμικά επίπεδα. Αυτό σημαίνει πως αλλάζοντας την κλίμακα των δεδομένων και κανονικοποιώντας, πρίν τα δώσουμε στον αλγόριθμό μας, μπορεί να αλλάξει την συμπεριφορά του αλγόριθμου.

Αυτό κάνουμε εδώ. Μπορείτε να διαβάσετε παραπάνω εδώ.


https://pytorch.org/vision/stable/transforms.html


In [ ]:
# Transformations applied on each image
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to tensors
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize with mean and std dev
])

train_dataset.transform = transform
test_dataset.transform = transform

### Data Loaders

Τα έχουμε ξαναδεί τα Data Loaders στο προηγούμενό μας Notebook.

Cut-and-paste από εκεί:


---


Στις μοντέρνες εφαρμογές, τα νευρωνικά δίκτυα συνήθως χρησιμοποιούνται σε σετ δεδομένων τεράστιου μεγέθους. Αυτά χρειάζονται προσεκτική μεταχείριση. Εμάς δεν μας αφορά άμεσα, μιά που όλα τα παραδείγματα που θα δούμε χωράνε την μνήμη (RAM) του επεξεργαστή που μας δείνει το Colab.

Αλλά επειδή έτσι λειτουργεί η PyTorch, πρέπει να χρησιμοποιήσουμε τα Data Loaders για τα δεδομένα μας.

Μπορούμε απλά να το σκεφτούμε σαν "black box" προς το παρόν.

Αλλιώς, δείτε εδώ:

* Τutorial: https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

* https://towardsdatascience.com/beginners-guide-to-loading-image-data-with-pytorch-289c60b7afec
* https://stackoverflow.com/questions/51756581/how-do-i-turn-a-pytorch-dataloader-into-a-numpy-array-to-display-image-data-with

* https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html

Για τις δικές μας ανάγκες προς το παρόν, αρκεί να ξέρουμε πως τα δεδομένα πρέπει να τα διαμορφώσουμε με ειδικό τρόπο, σε κάτι που λέγεται "**Data Loader**". Και αρκεί να ξέρουμε πως πρέπει να χρησιμοποιήσουμε κώδικα όπως εδώ, για να φτιάξουμε αυτά τα "Data Loaders".

---



In [ ]:
# Data loaders for training and testing
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


### Δεδομένα από τα Data Loaders

Μπορούμε να ζητάμε ένα-ένα τα δεδομένα από τα DataLoaders.

1. Δημιουργούμε ένα ``iterator`` από το dataloader με την εντολή
```
dataiter = iter(train_loader)
```
2. Τώρα με την εντολή ``next`` ζητάμε το επόμενο "batch" δεδομένων. Το μέγεθος του batch το ορίσαμε με την εντολή ``batch_size`` όταν δημιουργήσαμε το DataLoader (``batch_size = 32``).

Για να το δείτε αυτό στην πράξη, τρέξτε τον κώδικα:
```
images, lables = next(dataiter)
print(lables)
print(len(lables))
```

Εμείς χρησιμοποιούμε το ``dataiter`` για να δούμε τις πρώτες δέκα εικόνες. Εάν ξανατρέξετε τον ίδιο κώδικα, θα δείτε πως αλλάζουν οι εικόνες. Κάθε φορά που καλούμε το ``next(dataiter)`` η εντολή μας επιστρέφη το επόμενο batch δεδομένων.

In [ ]:
import matplotlib.pyplot as plt
classes = train_dataset.classes

k = 10
# Create an iterator from the dataloader
dataiter = iter(train_loader)
images, labels = next(dataiter)  # Use next() to get one batch of images and labels

plt.figure(figsize=(10, 2 * k))
for idx in range(min(k, len(images))):
  ax = plt.subplot(1, k, idx + 1)
  img = images[idx].numpy().transpose((1, 2, 0))  # Convert from PyTorch tensor to NumPy array and transpose
  img = (img - img.min()) / (img.max() - img.min())  # Normalize to [0, 1] for visualization

  plt.imshow(img, cmap='gray')
  plt.title(classes[labels[idx]])
  plt.axis('off')
plt.show()

## Αρχίζουμε με ένα απλό νευρωνικό

In [ ]:
# Simple Neural Network
class SimpleNN(nn.Module):
    def __init__(self, input_size = 784, output_size = 10):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, output_size)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten the input
        out = self.fc1(x)
        return out # Note: we do not need a softmax is using CrossEntropyLoss (see optimization function)

## Εκπαίδευση Νευρωνικών: ``model.fit(X,y)``

Στο [προηγούμενο Notebook](https://colab.research.google.com/drive/1hCXEaMOxXnxWPoPn9YYufDF2huHUgHAS?usp=sharing) είδαμε τις βασικές ίδεες για την εκπαίδευση των νευρωνικών δικτύων -- δηλαδή, πως βρίσκουμε παραμέτρους που ελαχιστοποιούν το σφάλμα στα δεδομένα εκπαίδευσης.

Ο κώδικας είναι πιο περίπλοκος. Η ιδέα είναι ίδια. Και αυτό το έχουμε ξαναδεί στο προηγούμενο Notebook. Επειδή είναι σημαντικό, τα ξαναγράφουμε εδώ:


Αντί για το απλό ```model.fit(X,y)``` χρειαζόμαστε μία διαδικασία που λέει πώς θα ψάξουμε και αρκετά καλές παραμέτρους, και πόσο θα ψάξουμε.

Ο παρακάτω κώδικας το κάνει αυτό:

```
train(model, train_loader, test_loader, optimizer, epochs)
```
Το

```model```

είναι το μοντέλο μας, το

```train_loader```

είναι τα δεδομένα εκπαίδευσης, το

```test_loader```

είναι τα δεδομένα εκτίμησης. Χρησιμοποιούμε το ``train_loader``, δηλαδή τα δεδομένα εκπαίδευσης, για την εκπαίδευση του αλγορίθμου. Χρησιμοποιούμε τα δεδομένα εκτίμησης από το ``test_loader`` για να εκτιμήσουμε το σφάλμα σε αυτά τα δεδομένα, αλλά σε αντίθεση με τα δεδομένα από το ``train_loader`` δεν τα χρησιμοποιούμε για να βελτιώσουμε τις παραμέτρους του μοντέλου.

```optimizer```

μας λέει πως θα ψάξουμε, και το

```epochs```

μας λέει πόσο χρόνο θα ψάξουμε -- συγκεκριμένα, πόσες φορές θα περάσουμε όλα τα δεδομένα μέχρι να βρούμε παραμέτρους που ταιριάζουν (αρκετά καλά) με τα δεδομένα εκπαίδευσης από το ``train_loader``.



### Τα βασικά βήματα της εκπαίδευσης

(Γράψαμε παρόμοια και στο [προηγούμενο Notebook](https://colab.research.google.com/drive/1hCXEaMOxXnxWPoPn9YYufDF2huHUgHAS?usp=sharing))

**Διαισθητικά**, ο παρακάτω κώδικας κάνει το εξής:
1. Ζητάει από τον Data Loader να του δώσει μερικά δείγματα από τα δεδομένα εκπαίδευσης.
2. Βλέπει τι τιμές ``y_pred`` προβλέπει το μοντέλο με τις παραμέτρους που έχει.
3. Εκτιμάει το σφάλμα μεταξύ του ``y_pred`` και ``y``.
4. Ψάχνει για μικρές αλλαγές που μπορούν να γίνουν στις παραμέτρους του μοντέλου, για να μειωθεί το σφάλμα. Μετά αλλάζει τις παραμέτρους του μοντέλου προς την κατεύθυνση αυτής τις αλλαγής.
5. Επαναλαμβάνει αυτήν την διαδικασία.

### Κάποιες λεπτομέρειες για τον παρακάτω κώδικα
1. ``device``: αυτό λέει στο Colab να χρησιμοποιήσει GPU ή CPU για την εκπαίδευση. Όποτε μπορούμε, χρησιμοποιούμε GPU για νευρωνικά δίκτυα.
2. ``model.to(device)``, ``x.to(device)``, ``y.to(device)``: αυτές οι εντολές μετακινούν τα δεδομένα και τις παραμέτρους του μοντέλου στον επεξεργαστή που θα χρησιμοποιήσουμε.
3. ``criterion = nn.CrossEntropyLoss()``: αυτό είναι το κριτήριο που χρησιμοποιείτε για τον υπολογισμού του σφάλματος (loss) μεταξύ του ``y_pred`` που προβλέπει το μοντέλο, και το ``y``. Το σφάλμα υπολογίζεται με την εντολή ``loss = criterion(outputs, y)``. Προσοχή: όταν χρησιμοποιούμε αυτή την συνάρτηση, **δεν χρειάζεται** να βάλουμε σαν τελευταίο επίπεδο το Softmax. Όταν χρησιμοποιούμε το Softmax, τότε χρησιμοποιούμε ``criterion = nn.NLLLoss``.
4. ``scheduler``: όταν βρεθεί μια κατεύθυνση που φαίνεται να βελτιώνει (μειώνει) το σφάλμα, πέρνουμε ένα βήμα προς αυτήν την κατεύθυνση. Το ``scheduler`` μας επιτρέπει να αλλάζουμε το μέγεθος του βήματος που παίρνουμε.

### Συνολικά, ο κώδικας:

1. Εκπαιδεύει το μοντέλο, προσπαθόντας να βρεί καλές παραμέτρους.
2. Αποθηκεύει το σφάλμα σε κάθε βήμα, και επίσης υπολογίζει την ακρίβεια σε κάθε βήμα. Σχετίζονται αυτά τα δύο (μικρότερο σφάλμα συνήθως αντιστοιχεί σε μεγαλύτερη ακρίβεια) αλλά η σχέση δεν είναι απόλυτη.
3. Δημιουργεί τα γραφήματα με τα σφάλματα και την ακρίβεια στα δεδομένα εκπαίδευσης, και στα δεδομένα εκτίμησης.


In [ ]:
from torch import optim

import torch
import copy
import torch.nn.functional as F
import matplotlib.pyplot as plt

def train(model, train_loader, test_loader, optimizer, epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = torch.nn.CrossEntropyLoss() # With this criterion, no softmax needed
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    train_losses = []
    test_losses = []
    test_accuracies = []  # List to store accuracy for each epoch
    best_accuracy = 0  # Best accuracy found
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()  # Update the learning rate

        train_losses.append(total_loss / len(train_loader))

        model.eval()
        total_test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_test_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_loss = total_test_loss / len(test_loader)
        epoch_accuracy = correct / total
        test_losses.append(epoch_loss)
        test_accuracies.append(epoch_accuracy)

        if epoch_accuracy > best_accuracy:
            best_accuracy = epoch_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        print(f"Epoch {epoch+1}/{epochs}, Training Loss: {train_losses[-1]}, Testing Loss: {test_losses[-1]}, Testing Accuracy: {epoch_accuracy:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    print(f"Loaded the best model from epoch {best_epoch} with Testing Accuracy: {best_accuracy:.4f}")

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs+1), train_losses, label='Training Loss')
    plt.plot(range(1, epochs+1), test_losses, label='Testing Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs+1), test_accuracies, label='Testing Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

    return # train_losses, test_losses, test_accuracies, best_accuracy




## Evaluate

Γράφουμε και ένα πρόγραμμα που χρησιμοποιεί το ```test_loader``` και μας βγάζει την ακρίβεια του μοντέλου μας.

In [ ]:
def evaluate(model, data_loader, device):
    model.eval()  # Set model to evaluate mode
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

In [ ]:
# Πόσες παραμέτρους έχει το μοντέλο μας
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### GPU

Εάν υπάρχει διαθέσιμο GPU πάντα θέλουμε να το χρησιμοποιήσουμε. Οι υπολογισμοί απαραίτητοι για την εκπαίδευση νευρωνικών εκτελούνται πολύ πιο γρήγορα σε GPU σε σχέση με CPU. (``Runtime``--> ``Change runtime type`` --> ``T4 GPU``)

In [ ]:
# set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
device

In [ ]:
torch.manual_seed(42)
model = SimpleNN()
model.to(device)

### Πόσες παραμέτρους έχει το μοντέλο

Το πρώτο μας μοντέλο έχει μόνο ένα γραμμικό, fully-connected επίπεδο. Έχουμε 28 x 28 = 784 features, και 10 outputs. Οπότε έχουμε 784 x 10 + 10 = 7840 + 10 = 7850 παραμέτρους.

Χρησιμοποιούμε την εντολή που γράψαμε πιο πάνω, για να επιβεβαιώσουμε τον υπολογισμό μας.

In [ ]:
count_trainable_parameters(model)

In [ ]:
torch.manual_seed(42)

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 10
train(model, train_loader, test_loader, optimizer, epochs)

# Καλά τα πήγε...

Ακόμα και το απλό νευρωνικό τα πήγε αρκετά καλά.
Το επόμενο μας βήμα είναι να χτίσουμε πιο βαθύ νευρωνικό δίκτυο χρησιμοποιώντας convolutional επίπεδα.

# CNN: Model Implementation.

Τώρα θα χτίσουμε το CNN -- το νευρωνικό με convolutional επίπεδα. Ενώ στο προηγούμενο Notebook χρησιμοποιούσαμε μόνο γραμμικά επίπεδα και ReLU, εδώ θα χρησιμοποιήσουμε

* Fully-connected επίπεδα `nn.Linear`
* Convolutional επίπεδα: `torch.nn.Conv2d`
* Relu επίπεδα: `nn.ReLU`
* Max Pooling επίπεδα:`max_pool2d(x, 2)`

Επίσης χρησιμοποιούμε την εντολή
```
x = x.view(-1, 320)
```
Αυτό δεν είναι επίπεδο -- απλά αλλάζει την δομή του $x$, και το κάνει ένα διάνυσμα που μπορεί να μπεί κατευθείαν στον πρώτο fully-connected επίπεδο, ``self.fc1 = nn.Linear(320, 50)``.

https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html


### Λεπτομέρειες για κάθε επίπεδο

Δίνουμε κάποιες λεπτομέρειες για το κάθε επίπεδο:

```
nn.Conv2d(1,10,kernel_size=5)
```
* Το πρώτο "1" μας λεει πως έχουμε ένα ``channel`` σαν input: αυτό το επίπεδο δέχεται σαν input μία εικόνα με ένα μόνο channel, δηλαδή grayscale (μαυρόασπρη).
* Το "10" μας λέει πως το output του επίπεδου έχει 10 channels.
* Το ``kernel_size = 5`` μας λέει πως έχουμε ένα kernel 5 επί 5, οπότε 25 παραμέτρους.
* Αφού δεν δίνουμε οδηγίες για ``zero-padding`` ή για ``stride``, έχουμε ``zero-padding = 0``, και ``stride = 1``.

Αυτό σημαίνει πως το output μας είναι 10 x 24 x 24.

```
x = F.max_pool2d(x,2)
```
Αυτό το επίπεδο είναι Max Pooling. Κοιτάει κάθε 2x2 κουτάκι σε κάθε channel ξεχωριστά, και το αντικαθιστά με την μέγιστη τιμή σε αυτούς τους 4 αριθμούς που περιέχει το 2x2 κουτάκι.

Οπότε το output μας μειώνεται σε μέγεθος και γίνεται: 10 x 12 x 12.

```
nn.Conv2d(10,20,kernel_size=5)
```
Τώρα έχουμε 10 input channels -- αναγκαστικά, για να ταιριάζει με το output του προηγούμενου convolutional επιπέδου -- και 20 output channels. Το kernel είναι πάλι 5 επί 5, και πάλι έχουμε ``zero-padding = 0``, και ``stride = 1``.

Αυτό σημαίνει πως το output μας είναι 20 x 8 x 8

Με το επόμενο
```
x = F.max_pool2d(x,2)
```
το output μας γίνεται 20 x 4 x 4.

Οπότε έχουμε 20 * 4 * 4 = 320 αριθμούς συνολικά. Για να μπούνε σε fully-connected επίπεδο, πρέπει να τις βάλει όλους σε ένα διάνυσμα. Αυτό ακριβώς κάνει η εντολή
```
x.view(-1,320)
```

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = F.max_pool2d(x, 2)

        x = self.conv2(x)
        x = self.relu(x)
        x = F.max_pool2d(x, 2)

        x = x.view(-1, 320)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


In [ ]:
torch.manual_seed(42)
model = ConvNet()
model.to(device)


### Πόσες παραμέτρους έχει το μοντέλο μας;

Το προηγούμενο είχε 7850. Αυτό το μοντέλο είναι πολύ πιο μεγάλο και πιο βαθύ. Αλλά επειδή χρησομοποιούμε convolutional επίπεδα, δεν έχει πολλές παραπάνω παράμετρους.

In [ ]:
count_trainable_parameters(model)

In [ ]:
torch.manual_seed(42)

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 10
train(model, train_loader, test_loader, optimizer, epochs)

##  Πώς τα πήγαμε;

In [ ]:
evaluate(model, test_loader, device)

# Τα πήγαμε πολύ καλά!

Αλλά δεν έχουμε 100% ακρίβεια. Αξίζει να δούμε που κάνει λάθος το μοντέλο μας. Μπορεί να μας δώσει κάποια ιδέα για το πως να το βελτιώσουμε, ή μπορεί να καταλάβουμε πως δεν υπάρχει καλύτερη λύση, όπως είδαμε με τα δεδομένα του διαβήτη σε [αυτό το Notebook](https://colab.research.google.com/drive/162hNnEDlmoAzcNFhEMQFmMXFQeVPHgHr?usp=sharing), που δεν θα υπήρχε νόημα να ψάξουμε για λύση με 100% ακρίβεια.

Η παρακάτω συνάρτηση

In [ ]:
import torch
import matplotlib.pyplot as plt

def show_misclassified_images(model, dataloader, classes, device):
    """
    Displays images that were misclassified by the model along with the true and predicted labels.

    Args:
    model (torch.nn.Module): The trained model.
    dataloader (torch.utils.data.DataLoader): DataLoader for the test dataset.
    classes (list): List of class names.
    device (str): Device to perform computations on ('cuda' or 'cpu').
    """
    model.eval()  # Set the model to evaluation mode
    model.to(device)

    with torch.no_grad():  # No need to track gradients
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            misclassified_indices = (predicted != labels)
            misclassified_images = images[misclassified_indices]
            misclassified_labels = labels[misclassified_indices]
            misclassified_preds = predicted[misclassified_indices]

            if misclassified_images.size(0) > 0:
                # Move images back to CPU for visualization
                misclassified_images = misclassified_images.cpu()
                misclassified_labels = misclassified_labels.cpu()
                misclassified_preds = misclassified_preds.cpu()

                plt.figure(figsize=(15, 5))
                for idx in range(min(misclassified_images.size(0), 10)):  # Display up to 10 misclassified images
                    ax = plt.subplot(2, 5, idx + 1)
                    img = misclassified_images[idx].numpy().transpose((1, 2, 0))
                    img = (img - img.min()) / (img.max() - img.min())  # Normalize to [0, 1] for visualization
                    plt.imshow(img, cmap='gray')
                    plt.title(f"True: {classes[misclassified_labels[idx]]}\nPred: {classes[misclassified_preds[idx]]}")
                    plt.axis('off')
                plt.show()

In [ ]:
device

In [ ]:
show_misclassified_images(model, test_loader, test_dataset.classes, device)

## Άσκηση

Παίξτε με την εντολή εκπαίδευσης
```
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 10
train(model, train_loader, test_loader, optimizer, epochs)
```
να δείτε πως αλλάζουν τα πράγματα. Αλλάξτε, για παράδειγμα, το ``lr=0.001``. Αυτή η παράμετρος ελέγχει πόσο μεγάλο "βήμα" παίρνει ο αλγόριθμος, δηλαδή, πόσο πρόοδο κάνει. Εάν είναι πολύ μικρός αυτός ο αριθμός, μπορεί η βελτίωση του μοντέλου να είναι πολύ αργή. Εάν είναι πολύ μεγάλος αυτός ο αριθμός, μπορεί η εκπαίδευση να μην συγκλίνει και να μήν βρεθούν ποτέ καλές παραμέτρους.